# Suite CNT — Containers

A container holds elements by index (`Schemas.OfContainer`): a property's value, read and written through
`Visitors.OfContainer`, its elements each through a `Visitors.OfAny`. Dense, its elements are a packed array; sparse,
blocks of them; with neither, it is dense for an integer-like index and a dict for any other hashable one. These cases
cover proxies, iterators, the value objects containers hold, and snapshots.

In [ ]:
import { JSON, Plain, Proxies, Schemas as S, YAML } from "@mbse/schemas/Framework";
import { DecodeError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, raises, root_of, same, text } from "./support.js";

const Int = new S.OfScalar.Data(BigInt);
const Text = new S.OfScalar.Data(String);
const Float = new S.OfScalar.Data(Number);
const Phone = new S.OfObject.Builder().properties(text("number"), text("label")).create();
const Cell = new S.OfObject.Builder().properties(text("row", BigInt), text("col", BigInt)).create();
const Tags = new S.OfContainer.Builder().index(Int).element(Text).create(); // dense
const Phones = new S.OfContainer.Builder().index(Text).element(Phone).create(); // a dict
const Grid = new S.OfContainer.Builder().index(Cell).element(Float).create(); // dense, of two indices
const Sparse = new S.OfContainer.Builder().index(Int).element(Text).sparse().create();
const Rows = new S.OfContainer.Builder().index(Int).element(Tags).create(); // nested
const Card = new S.OfObject.Builder().name("Card").ref(String.raw`card-(\d+)`).properties(
  (p) => p.name("tags").of(Tags), (p) => p.name("phones").of(Phones), (p) => p.name("grid").of(Grid),
  (p) => p.name("sparse").of(Sparse), (p) => p.name("rows").of(Rows), text("name")).create();
const store = new Proxies.OfStore();
store.register(Card);
const B: any = store;

/** The container a property holds, as its visitor; null where it holds none (its visitor then visits nothing). */
function container(read: any): any {
  const found: any[] = [];
  read.as_value((a: any) => a.as_container((c: any) => found.push(c) && c));
  return found[0].as_schema() !== null ? found[0] : null;
}

/** An element's scalar, or null. */
function valueOfAny(e: any): unknown {
  const found: unknown[] = [];
  e.as_scalar((s: any) => found.push(s.get()) && s);
  return found.length > 0 ? found[0] : null;
}

/** A container's elements present, each with its index, as scalars. */
function scalars(c: any): unknown[] {
  const out: unknown[] = [];
  c.as_entries((pairs: Iterable<[unknown, any]>) => {
    for (const [index, e] of pairs) out.push([index, valueOfAny(e)]);
  });
  return out;
}

/** Each position of a dense or sparse container, with its element's scalar, by its iterator. */
function walk(c: any): unknown[] {
  const out: unknown[] = [];
  c.as_ordered((it: any) => {
    it.first();
    while (it.index() !== null) {
      const values: unknown[] = [];
      it.as_item((e: any) => e.as_scalar((s: any) => values.push(s.get()) && s));
      out.push([it.index(), values[0]]);
      it.next();
    }
  });
  return out;
}

/** Writes through a property's container. */
const into = (cb: (c: any) => unknown) => (p: any) => p.as_value((a: any) => a.as_container(cb));

## CNT-01 · Elements by index: written through a builder's container, read through a data instance's; reads never raise

In [ ]:
let card = B.Card()
  .tags(into((t) => t.at(0n).as_scalar("a") && t.at(2n).as_scalar("c")))
  .phones(into((d) => d.at("home").as_object((r: any) => r.number("1"))))
  .grid(into((g) => g.at([1n, 2n]).as_scalar(3.5)))
  .create();
{
  const [tags, phones, grid] = [container(card.tags), container(card.phones), container(card.grid)];
  assert(tags.as_schema() === Tags && same([tags.length, phones.length, grid.length], [3, 1, 6])); // dense: every index up to the last
  assert(equal(scalars(tags), [[0n, "a"], [2n, "c"]]) && valueOfAny(tags.at(1n)) === null); // an index with nothing reads as empty
  assert(equal(walk(tags), [[0n, "a"], [1n, null], [2n, "c"]])); // an iterator visits every position of a dense container
  assert(equal(walk(grid), [[[0n, 0n], null], [[0n, 1n], null], [[0n, 2n], null], [[1n, 0n], null], [[1n, 1n], null], [[1n, 2n], 3.5]]));
  const seen: unknown[] = [];
  phones.as_dict((d: Map<unknown, unknown>) => seen.push([...d.keys()].sort()));
  tags.as_dict(() => seen.push("not a dict")); // not called: tags is dense
  phones.as_ordered(() => seen.push("not ordered")); // not called: phones is a dict
  assert(equal(seen, [["home"]]));
  phones.at("home").as_object((r: any) => seen.push(`${r.number}`) && r);
  assert(seen.at(-1) === "1");
  assert(container(B.Card().create().tags) === null); // not set
  card.name.as_value((a: any) => a.as_container((c: any) => seen.push([c.as_schema(), c.length, c.at(0n).as_schema()])));
  assert(equal(seen.at(-1), [null, 0, null])); // a value of another kind: an empty visitor, read as an empty container
  card.name.as_value((a: any) => a.as_container((c: any) => c.at(0n).as_container((x: any) => seen.push(x.as_schema()))));
  assert(seen.at(-1) === null);
  const empty: unknown[] = [];
  card.name.as_value((a: any) => a.as_container((c: any) => c.as_entries((pairs: Iterable<unknown>) => empty.push([...pairs]))
    .as_ordered(() => empty.push("x")).as_dict(() => empty.push("x"))));
  assert(equal(empty, [[]]));
  raises(ReadOnlyError, () => tags.remove(0n), "the container is read-only; write through a builder");
  raises(ReadOnlyError, () => tags.at(0n).as_scalar("z"), "the container[0] is read-only; write through a builder");
  raises(ReadOnlyError, () => card.tags.as_value((a: any) => a.as_container(tags)), "property 'tags' is read-only; write through a builder");
  raises(ReadOnlyError, () => B.Card().name((p: any) => p.as_value((a: any) => a.as_container(tags))), "property 'name' does not hold a container");
  assert(repr(tags) === "<container of 2>");
  // A builder edits a copy, which it reads back; update() writes it, clone() copies it.
  const edited = B.Card(card).tags(into((t) => {
    t.remove(2n).at(1n).as_scalar("b");
    assert(t.length === 2);
  })).update();
  assert(edited === card && equal(scalars(container(card.tags)), [[0n, "a"], [1n, "b"]]));
  const copied = B.Card(card).clone();
  assert(equal(scalars(container(copied.tags)), [[0n, "a"], [1n, "b"]]) && container(copied.tags) !== container(card.tags));
  // A container given as a value is written as it is; one of another schema is refused.
  const given = B.Card().tags((p: any) => p.as_value((a: any) => a.as_container(container(card.tags)))).create();
  assert(equal(scalars(container(given.tags)), [[0n, "a"], [1n, "b"]]));
  raises(TypeError, () => B.Card().tags((p: any) => p.as_value((a: any) => a.as_container(container(card.grid)))),
    "property 'tags' takes a container of its schema or a Spec, got _ContainerData");
  assert(equal(scalars(container(B.Card().tags(container(card.tags)).create().tags)), [[0n, "a"], [1n, "b"]])); // as a property's value
  const twice = B.Card().tags(into((t) => t.at(0n).as_scalar("a"))).tags(into((t) => t.at(1n).as_scalar("b"))).create();
  assert(equal(scalars(container(twice.tags)), [[0n, "a"], [1n, "b"]])); // a builder reads back, and adds to, what it has written
  // Containers nest: a container's element may be a container.
  const nested = B.Card().rows(into((r) => r.at(1n).as_container((t: any) => t.at(0n).as_scalar("x")))).create();
  const inner: unknown[] = [];
  container(nested.rows).at(1n).as_container((t: any) => inner.push(scalars(t)));
  assert(equal(inner, [[[0n, "x"]]]) && container(nested.rows).length === 2);
}

## CNT-02 · Iterators: positions before the first, at each, and past the last; a builder's extends a container and inserts into one of int indices

In [ ]:
{
  const sparse = B.Card().sparse(into((c) => c.at(10n).as_scalar("j") && c.at(11n).as_scalar("k") && c.at(40n).as_scalar("z"))).create();
  assert(equal(walk(container(sparse.sparse)), [[10n, "j"], [11n, "k"], [40n, "z"]]) && container(sparse.sparse).length === 3);
  const moves: unknown[] = [];
  container(sparse.sparse).as_ordered((it: any) => moves.push(it.index(), it.first().index(), it.next().index(), it.last().index(),
    it.next().index(), it.next().index(), it.prev().index(), it.first().prev().index(), it.prev().index(), String(it)));
  assert(equal(moves, [null, 10n, 11n, 40n, null, null, 40n, null, null, "<iterator at None>"]));
  const calls: unknown[] = [];
  container(sparse.sparse).as_ordered((it: any) => it.last().next().as_item((e: unknown) => calls.push(e)));
  assert(calls.length === 0); // past the end of a read-only container: not called
  raises(ReadOnlyError, () => container(sparse.sparse).as_ordered((it: any) => it.first().prepend("x")),
    "the container is read-only; write through a builder");

  /** Edits a card's container through its builder's iterator. */
  const edit = (card: any, name: string, cb: (it: any) => unknown) =>
    B.Card(card)[name](into((c) => c.as_ordered(cb))).update();

  edit(sparse, "sparse", (it) => it.last().next().as_item((e: any) => e.as_scalar("past")).first().prev().as_item(
    (e: any) => e.as_scalar("before")));
  assert(equal(walk(container(sparse.sparse)), [[10n, "before"], [11n, "j"], [12n, "k"], [41n, "z"], [42n, "past"]]));
  edit(sparse, "sparse", (it) => it.first().append("after").prepend((e: any) => e.as_scalar("in")));
  assert(equal(walk(container(sparse.sparse)), [[10n, "before"], [11n, "in"], [12n, "after"], [13n, "j"], [14n, "k"], [43n, "z"], [44n, "past"]]));
  const fresh = B.Card().tags(into((c) => c.as_ordered(
    (it: any) => it.as_item((e: any) => e.as_scalar("first")).next().as_item((e: any) => e.as_scalar("second"))))).create();
  assert(equal(walk(container(fresh.tags)), [[0n, "first"], [1n, "second"]])); // an empty container extended from its ends
  edit(fresh, "tags", (it) => it.last().next().append("end").first().prev().prepend("start"));
  assert(equal(walk(container(fresh.tags)), [[0n, "start"], [1n, "first"], [2n, "second"], [3n, "end"]]));
  const blank = B.Card().tags(into((c) => c.as_ordered((it: any) => it.append("only")))).create();
  assert(equal(walk(container(blank.tags)), [[0n, "only"]]));
  const ends = B.Card().tags(into((c) => c.as_ordered(
    (it: any) => it.first().as_item((e: any) => e.as_scalar("a")).last().next().prepend("z").first().prev().append("y")))).create();
  assert(equal(walk(container(ends.tags)), [[0n, "y"], [1n, "a"], [2n, "z"]])); // at the ends, of an empty container and of one with elements
  const lone = B.Card().tags(into((c) => c.as_ordered((it: any) => it.prepend("p")))).create();
  assert(equal(walk(container(lone.tags)), [[0n, "p"]]));
  const gridded = B.Card(card).grid(into((g) => g.as_ordered((it: any) => it.first().as_item((e: any) => e.as_scalar(0.5))))).update();
  assert(equal(walk(container(gridded.grid))[0], [[0n, 0n], 0.5]));
  raises(ValueError, () => edit(card, "grid", (it) => it.first().append(1.0)),
    "property 'grid': only a dense or sparse container of int indices has positions to insert at; write through at(index)");
}

## CNT-03 · Value objects in containers belong to the object holding the container: edited in place, copied by a clone, removed with their element

In [ ]:
{
  const owner = B.Card().phones(into((d) => d.at("home").as_object((r: any) => r.number("1").label("h"))
    && d.at("work").as_object((r: any) => r.number("2")))).create();
  const held = new Map<unknown, any>();
  container(owner.phones).as_dict((d: Map<unknown, any>) => d.forEach((e, k) => held.set(k, e)));
  const home: any[] = [];
  held.get("home").as_object((r: any) => home.push(r) && r);
  assert(Proxies.owner_of(home[0]) === owner && `${home[0].label}` === "h");
  B.Card(owner).phones(into((d) => d.at("home").as_object((r: any) => r.label("x")))).update();
  const again: any[] = [];
  container(owner.phones).at("home").as_object((r: any) => again.push(r) && r);
  assert(again[0] === home[0] && `${home[0].label}` === "x" && `${home[0].number}` === "1"); // edited in place
  const twin = B.Card(owner).clone();
  const copied: any[] = [];
  container(twin.phones).at("home").as_object((r: any) => copied.push(r) && r);
  assert(copied[0] !== home[0] && Proxies.owner_of(copied[0]) === twin && `${copied[0].number}` === "1");
  B.Card(owner).phones(into((d) => d.remove("home"))).update();
  assert(container(owner.phones).length === 1 && Proxies.owner_of(home[0]) === owner);
  const Kit = new S.OfObject.Builder().properties((p) => p.name("parts").of(Tags)).create(); // a value object holding a container
  const Box = new S.OfObject.Builder().name("Box").ref().properties((p) => p.name("kit").of(Kit)).create();
  store.register(Box);
  const box = B.Box().kit((p: any) => p.as_object((k: any) => k.parts(into((t) => t.at(0n).as_scalar("bolt"))))).create();
  const twinBox = B.Box(box).clone();
  const parts: unknown[] = [];
  twinBox.kit.as_object((k: any) => {
    k.parts.as_value((a: any) => a.as_container((t: any) => parts.push(scalars(t))));
    return k;
  });
  assert(equal(parts, [[[0n, "bolt"]]])); // copied with the value object that holds it
  const moved = B.Card().tags(into((t) => t.at(0n).as_scalar("a"))).create();
  B.Card(moved).tags(into((t) => t.as_ordered((it: any) => it.first().prepend("z")))).update();
  assert(equal(walk(container(moved.tags)), [[0n, "z"], [1n, "a"]]));
}

## CNT-04 · Snapshots: a dense container as an array, nested or flat for several indices, a sparse one as blocks, a dict by text as a mapping, entries otherwise; read back, and over what a store holds

In [ ]:
card = B.Card()
  .tags(into((t) => t.at(0n).as_scalar("a") && t.at(2n).as_scalar("c")))
  .phones(into((d) => d.at("home").as_object((r: any) => r.number("1"))))
  .grid(into((g) => g.at([1n, 2n]).as_scalar(3.5) && g.at([0n, 0n]).as_scalar(0.25)))
  .sparse(into((s) => s.at(3n).as_scalar("x") && s.at(4n).as_scalar("y") && s.at(9n).as_scalar("z")))
  .rows(into((r) => r.at(0n).as_container((t: any) => t.at(1n).as_scalar("n"))))
  .create();
const written = root_of(Plain.ToPlain(store).OfObject(Card, card));
assert(container(card.grid).length === 6); // the box its indices span
assert(equal(written, map({ tags: ["a", null, "c"], phones: { home: { number: "1" } }, grid: [[0.25, null, null], [null, null, 3.5]],
  sparse: [{ at: 3n, items: ["x", "y"] }, { at: 9n, items: ["z"] }], rows: [[null, "n"]] })));
const flat = root_of(Plain.ToPlain(store, { flat: true }).OfObject(Card, card));
assert(equal(flat.get("grid"), map({ shape: [2n, 3n], items: [0.25, null, null, null, null, 3.5] })));
assert(JSON.ToJSON(store, { flat: true }).OfObject(Card, card).includes('"grid": {"shape": [2, 3]')
  && YAML.ToYAML(store, { flat: true }).OfObject(Card, card).includes("shape:"));
const other = new Proxies.OfStore();
other.register(Card);
for (const snapshot of [Plain.ToPlain(store).OfObject(Card, card), Plain.ToPlain(store, { flat: true }).OfObject(Card, card)]) {
  const back = Plain.FromPlain(other).OfObject(Card, snapshot) as any; // the loader takes either form
  assert(equal(root_of(Plain.ToPlain(other).OfObject(Card, back)), written));
}
assert(Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) === card); // reloading: an update
assert(equal(root_of(Plain.ToPlain(store).OfObject(Card, card)), written));
// Any other index is written as entry-like elements: a dict by bytes, a dense container by text, a sparse one by text.
const ByBytes = new S.OfContainer.Builder().index(new S.OfScalar.Data(Uint8Array)).element(Int).create();
const Sorted = new S.OfContainer.Builder().index(Text).element(Int).dense().create();
const Keyed = new S.OfContainer.Builder().index(Cell).element(Int).sparse().create();
const Mixed = new S.OfObject.Builder().properties(text("k"), text("n", BigInt)).create();
const ByMixed = new S.OfContainer.Builder().index(Mixed).element(Int).create();
const Note = new S.OfObject.Builder().name("Note").ref().properties((p) => p.name("b").of(ByBytes), (p) => p.name("s").of(Sorted),
  (p) => p.name("k").of(Keyed), (p) => p.name("m").of(ByMixed)).create();
store.register(Note);
const note = B.Note().b(into((c) => c.at(new Uint8Array([1])).as_scalar(1n)))
  .s(into((c) => c.at("b").as_scalar(2n) && c.at("a").as_scalar(1n)))
  .k(into((c) => c.at([0n, 1n]).as_scalar(1n) && c.at([0n, 2n]).as_scalar(2n) && c.at([3n, 0n]).as_scalar(3n)))
  .m(into((c) => c.at(["x", 1n]).as_scalar(5n))).create();
const noted = root_of(Plain.ToPlain(store).OfObject(Note, note));
assert(equal(noted, map({ b: [{ index: "AQ==", element: 1n }], s: [{ index: "a", element: 1n }, { index: "b", element: 2n }],
  k: [{ at: { row: 0n, col: 1n }, items: [1n, 2n] }, { at: { row: 3n, col: 0n }, items: [3n] }],
  m: [{ index: { k: "x", n: 1n }, element: 5n }] })));
other.register(Note);
assert(equal(root_of(Plain.ToPlain(other).OfObject(Note, Plain.FromPlain(other).OfObject(Note, Plain.ToPlain(store).OfObject(Note, note)) as any)), noted));
// A union's branch, or an intersection's part, may hold a container.
const Either = new S.OfUnion.Builder().branches((b) => b.name("tags").of(Tags), (b) => b.name("word").of(Text)).create();
const Choice = new S.OfObject.Builder().name("Choice").ref().properties((p) => p.name("pick").of(Either)).create();
store.register(Choice);
other.register(Choice);
const choice = B.Choice().pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags((t: any) => t.at(1n).as_scalar("q"))))).create();
const picked = root_of(Plain.ToPlain(store).OfObject(Choice, choice));
assert(equal(picked, map({ pick: { tags: [null, "q"] } })));
assert(equal(root_of(Plain.ToPlain(other).OfObject(Choice, Plain.FromPlain(other).OfObject(Choice, Plain.ToPlain(store).OfObject(Choice, choice)) as any)), picked));
B.Choice(choice).pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags((t: any) => t.at(2n).as_scalar("r"))))).update();
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, choice)), map({ pick: { tags: [null, "q", "r"] } }))); // its container edited
const sameBranch = B.Choice().pick((p: any) => p.as_value((a: any) => a.as_union(
  (u: any) => u.tags((t: any) => t.at(0n).as_scalar("p")).tags((t: any) => t.at(1n).as_scalar("q"))))).create(); // read back, then written again
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, sameBranch)), map({ pick: { tags: ["p", "q"] } })));
const givenBranch = B.Choice().pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags(container(card.tags))))).create();
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, givenBranch)), map({ pick: { tags: ["a", null, "c"] } }))); // a container given
raises(TypeError, () => B.Choice().pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags(1n)))),
  "branch 'tags' takes a container of its schema or a Spec, got int");
raises(ReadOnlyError, () => container(card.tags)((t: unknown) => t), "the container is read-only; write through a builder");
// An entry's property may hold a container: entries equal by their containers are one, in a set.
const Marks = new S.OfObject.Builder().name("Marks").ref().relations((r) => r.name("sets").me("owner").of(
  (rel) => rel.links("owner").properties((p) => p.name("tags").of(Tags), (p) => p.name("grid").of(Grid)).unique())).create();
store.register(Marks);
other.register(Marks);
const two = (t: any) => t.at(0n).as_scalar("a") && t.at(1n).as_scalar("b");
const cell = (g: any) => g.at([0n, 1n]).as_scalar(1.5);
const marks = B.Marks().sets((adj: any) => adj.insert((e: any) => e.tags(into(two)).grid(into(cell)))
  .insert((e: any) => e.tags(into(two)).grid(into(cell)))).create();
assert(marks.sets.length === 1 && equal(root_of(Plain.ToPlain(store).OfObject(Marks, marks)),
  map({ sets: [{ tags: ["a", "b"], grid: [[null, 1.5]] }] })));
const doubled = map({ root: "Marks/m", objects: { "Marks/m": { sets: [{ tags: ["a", "b"], grid: [[null, 1.5]] },
  { tags: ["a", "b"], grid: [[null, 1.5]] }] } } });
assert((Plain.FromPlain(other).OfObject(Marks, doubled) as any).sets.length === 1); // read once, by value
assert(container(note.s).length === 2); // a dense container by text: the elements present
const emptied = Plain.FromPlain(other).OfObject(Note, map({ root: "Note/e", objects: { "Note/e": { b: [{ index: "AQ==" }], s: [] } } })) as any;
assert(equal(root_of(Plain.ToPlain(other).OfObject(Note, emptied)), map({ b: [], s: [] }))); // an element absent; empty containers
const branch: any[] = [];
choice.pick.as_value((a: any) => a.as_union((u: any) => branch.push(u.tags) && u));
B.Choice().create().pick.as_value((a: any) => a.as_union((u: any) => branch.push(u.tags) && u));
assert(branch[0].length === 3 && branch[1].as_schema() === null); // a union value not set: its branch reads as empty

## CNT-05 · Malformed containers in snapshots are refused, with a path

In [ ]:
{
  const refused = (name: string, value: unknown, message: string) =>
    raises(DecodeError, () => Plain.FromPlain(store).OfObject(Card, map({ root: "Card/card-900", objects: { "Card/card-900": { [name]: value } } })), message);
  const at = '$.objects["Card/card-900"]';
  refused("tags", { a: 1n }, `${at}.tags: a dense container must be a list, got dict`);
  refused("tags", [1n], `${at}.tags[0]: expected str, got int`);
  refused("grid", [[1.0], 2.0], `${at}.grid[1]: a container of 2 indices nests 2 lists`);
  refused("grid", { shape: [2n], items: [1.0] }, `${at}.grid: a flat container's shape is 2 sizes, and its items as many as they make`);
  refused("grid", { shape: [1n, 2n], items: [1.0] }, `${at}.grid: a flat container's shape is 2 sizes`);
  refused("grid", { shape: "2, 3", items: [] }, `${at}.grid: a flat container's shape is 2 sizes`);
  refused("grid", { shape: [1n, 1n], items: [true] }, `${at}.grid.items[0]: expected float, got bool`);
  refused("sparse", { at: 1n }, `${at}.sparse: a sparse container must be a list of blocks, got dict`);
  refused("sparse", [{ at: 1n }], `${at}.sparse[0]: a block is {'at': index, 'items': [element, ...]}`);
  refused("sparse", [{ at: "1", items: [] }], `${at}.sparse[0].at: expected int, got str`);
  refused("phones", [], `${at}.phones: a container keyed by text must be a mapping, got list`);
  refused("phones", { home: 1n }, `${at}.phones.home: a value object must be a mapping, got int`);
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, map({ root: "Note/n", objects: { "Note/n": { m: {} } } })),
    '$.objects["Note/n"].m: a container must be a list, got dict');
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, map({ root: "Note/n", objects: { "Note/n": { m: [{ element: 1n }] } } })),
    `$.objects["Note/n"].m[0]: an entry of a container is {'index': index, 'element': element}`);
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, map({ root: "Note/n", objects: { "Note/n": { k: [{ at: { row: 1n }, items: [1n] }] } } })),
    `$.objects["Note/n"].k[0].at: an index has every property of its value object; ['col'] are not set`);
}
const Deep = new S.OfObject.Builder().properties((p) => p.name("cell").of(Cell)).create();
const Odd = new S.OfContainer.Builder().index(Deep).element(Int).create(); // entries: its index is neither ordered nor hashable
const Rare = new S.OfObject.Builder().name("Rare").ref().properties((p) => p.name("odd").of(Odd)).create();
store.register(Rare);
raises(DecodeError, () => Plain.FromPlain(store).OfObject(Rare, map({ root: "Rare/r", objects: { "Rare/r": { odd: [{ index: {}, element: 1n }] } } })),
  `$.objects["Rare/r"].odd[0].index: a container whose index is neither ordered nor hashable is not read from snapshots yet`);

## CNT-06 · An index is checked against the index type: a scalar of its type, or a tuple of a value object index's properties; a dense container's is not negative

In [ ]:
{
  const at = (name: string, index: unknown) => B.Card()[name](into((c) => c.at(index)));
  raises(TypeError, () => at("tags", "0"), "property 'tags': an index is a int, got str");
  raises(TypeError, () => at("tags", true), "property 'tags': an index is a int, got bool");
  raises(TypeError, () => at("grid", [1n]), "property 'grid': an index is a tuple of int, int, got [1]");
  raises(TypeError, () => at("grid", [1n, "2"]), "property 'grid': an index is a tuple of int, int, got [1, '2']");
  raises(ValueError, () => at("tags", -1n), "property 'tags': a dense container's index is not negative, got -1");
  raises(ValueError, () => at("grid", [0n, -1n]), "property 'grid': a dense container's index is not negative, got [0, -1]");
  B.Card().sparse(into((c) => c.at(-5n).as_scalar("ok"))); // sparse: any int
  const deep: any[] = [];
  B.Rare().odd(into((c) => deep.push(c)));
  raises(TypeError, () => deep[0].at(1n), "property 'odd': an index is a value of its index type, got int");
  // An index of a value object that is neither ordered nor hashable: given as a value object, written as one.
  const KeyedBy = new S.OfObject.Builder().name("Keyed").ref().properties((p) => p.name("key").of(Deep), (p) => p.name("odd").of(Odd)).create();
  store.register(KeyedBy);
  let keyed = B.Keyed().key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(2n))))).create();
  const key: any[] = [];
  keyed.key.as_object((k: any) => key.push(k) && k);
  keyed = B.Keyed(keyed).odd(into((c) => c.at(key[0]).as_scalar(7n))).update();
  assert(equal(root_of(Plain.ToPlain(store).OfObject(KeyedBy, keyed)).get("odd"), map([{ index: { cell: { row: 1n, col: 2n } }, element: 7n }])));
}